In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Pooling

The layers of the previous sections answer local questions. The edge
detector of that section reports, position by position, whether an
edge lies there. A classifier must answer a question about the whole image:
does this Fashion-MNIST picture show a shirt or a sneaker? This section asks
how a network gets from many local answers to one global answer. Two
obstacles stand in the way.

The first obstacle is reach. Some unit near the top of the network must have a
receptive field, the input region that can influence it
(that section), that covers the whole image. A $3\times3$
convolution with stride 1 widens the receptive field by two pixels, so after
$L$ such layers a unit sees a $(2L+1)\times(2L+1)$ patch, and covering a
$28\times28$ Fashion-MNIST image takes fourteen layers. If every second layer
is followed by a $2\times2$ step that halves the resolution, each later kernel
reaches twice as far in the input (that section), and each
$2\times2$ window adds its own extent (the equation with
$k=s=2$). Six $3\times3$ convolutions with two such steps then see a
$32\times32$ patch.

The second obstacle appears at the bottom of the network. The edge detector
responds at the pixel where the edge lies. Shift the
image by one pixel, as happens whenever the object or the camera moves
slightly, and the response moves by one pixel too. A classifier's answer
should not change because of such a shift.

A *pooling layer* addresses both obstacles. It replaces each small window of
its input by one summary value, the maximum or the average, and has no
parameters. With windows of size 2 and stride 2 it halves the resolution, and
the maximum of a window does not change when a response moves within it. The
shape rules of padding and stride (that section) carry over; only
the value used for padding changes. The network of the next section, LeNet,
alternates convolutions with pooling.

In [ ]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

In [ ]:
from functools import partial

## Maximum Pooling and Average Pooling

A pooling operator slides a window of fixed shape, the *pooling window*, over
its input, like the kernel of a convolution. At each position it computes one
output from the entries under the window. Unlike a convolution, it has no
kernel and therefore no parameters: *maximum pooling* (*max-pooling* for
short) outputs the largest entry in the window, and *average pooling* outputs
their mean.

Average pooling dates to the earliest CNNs. It is a form of image
downsampling that, instead of keeping every second pixel, averages
neighboring pixels and so smooths out variation from pixel to pixel.
Max-pooling was proposed in cognitive neuroscience as a model of how the
visual cortex might aggregate evidence hierarchically for object recognition
[@Riesenhuber.Poggio.1999], and an earlier version had been used in
speech recognition [@Yamaguchi.Sakamoto.Akabane.ea.1990]. The
Neocognitron [@Fukushima.1980] summarized neighborhoods earlier still.
It alternated layers of feature detectors with layers of complex cells. A
complex cell responded when one detector fired anywhere in a small
neighborhood, so its output tolerated small shifts of the input. In the
networks of the 2010s, max-pooling became the usual choice between
convolutions, and average pooling the usual choice at the end of a network.

![Max-pooling with a pooling window shape of $2\times 2$. The shaded portions are the first output element as well as the input tensor elements used for the output computation: $\max(0, 1, 3, 4)=4$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/pooling.svg)

In the figure a $2\times2$ window slides over a $3\times3$ input
with stride 1, from the upper left, left to right and top to bottom. It fits
at two positions along each axis, so the output is $2\times2$, and each output
entry is the maximum of the four entries under the window:

$$
\begin{aligned}
\max(0, 1, 3, 4) &= 4,\\
\max(1, 2, 4, 5) &= 5,\\
\max(3, 4, 6, 7) &= 7,\\
\max(4, 5, 7, 8) &= 8.
\end{aligned}
$$

Each output is the largest entry in its window. Small changes to the entries
that are the largest in no window, here 0, 1, 2, 3, and 6, leave the output
unchanged. A pooling layer with a $k_\textrm{h} \times k_\textrm{w}$ window
works the same way.

The function `pool2d` implements the forward computation. It has the loop
structure of `corr2d` in that section, with the multiply-and-sum
over the window replaced by a maximum or a mean. The argument `stride` moves
the window by more than one position at a time; we need it below.

In [2]:
def pool2d(X, pool_size, mode='max', stride=1):
    k_h, k_w = pool_size
    reduce = tf.reduce_max if mode == 'max' else tf.reduce_mean
    # Build Y from its entries, so that gradients flow through it
    return tf.stack([
        tf.stack([reduce(X[i: i + k_h, j: j + k_w])
                  for j in range(0, X.shape[1] - k_w + 1, stride)])
        for i in range(0, X.shape[0] - k_h + 1, stride)])

Applied to the input of the figure, `pool2d` should return
4, 5, 7, and 8.

In [3]:
X = d2l.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
pool2d(X, (2, 2))

<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[4., 5.],
       [7., 8.]], dtype=float32)>

Average pooling of the same input returns the four window means, 2, 3, 5,
and 6. Each window holds the numbers $a, a+1, a+3, a+4$ for some $a$, so its
mean $a+2$ lies 2 below its maximum $a+4$.

In [4]:
pool2d(X, (2, 2), 'avg')

<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[2., 3.],
       [5., 6.]], dtype=float32)>

## Tolerance to Small Shifts

Pooling with a window of $2\times2$ and stride 2 splits its input into
disjoint $2\times2$ blocks and reduces each block to one number. Consider a
feature map with a single response of value 1, such as the output of an edge
detector at an isolated edge, and move the response by one pixel at a time.
We compute the max-pooled and the average-pooled outputs for three positions
of the response in row 1 of a $4\times4$ map.

In [5]:
def single_response(i, j, n=4):
    """An n x n feature map that is 1 at (i, j) and 0 elsewhere."""
    return d2l.tensor([[float(r == i and c == j) for c in range(n)]
                       for r in range(n)])

for j in range(3):
    Z = single_response(1, j)
    print(f'response at (1, {j}): max',
          d2l.numpy(pool2d(Z, (2, 2), stride=2)).tolist(),
          'avg', d2l.numpy(pool2d(Z, (2, 2), 'avg', stride=2)).tolist())

response at (1, 0): max [[1.0, 0.0], [0.0, 0.0]] avg [[0.25, 0.0], [0.0, 0.0]]
response at (1, 1): max [[1.0, 0.0], [0.0, 0.0]] avg [[0.25, 0.0], [0.0, 0.0]]
response at (1, 2): max [[0.0, 1.0], [0.0, 0.0]] avg [[0.0, 0.25], [0.0, 0.0]]


Moving the response from column 0 to column 1 changes neither output: both
positions lie in the same block. Moving it on to column 2 crosses into the
next block, and the output moves with it. A one-pixel shift of a single
response is therefore absorbed when the response stays inside its block,
which happens for half of all horizontal positions, and it moves the pooled
output by one position otherwise. Pooling makes a representation *tolerant*
to some small shifts. It does not make it invariant to them.

The two kinds of pooling differ in what they report. Max-pooling reports
the response at full strength, 1, wherever it falls in the block: it answers
whether the feature is present somewhere in the window. Average pooling
reports 0.25, because the response covers one of the four positions: it
measures how much of the window the feature fills, so a strong but isolated
response is diluted by its neighbors.

A stride of 2 keeps one value per block, so the output can jump under a
one-pixel shift, as at the block boundary above. In the extreme case, keeping
every second pixel of a pattern that alternates from pixel to pixel,
$1, 0, 1, 0, \ldots$, gives all ones or all zeros, depending on a shift by
one pixel. This error, which arises when a signal changes faster than the
samples can follow, is called *aliasing*. It affects every stride-2
downsampler, pooled or convolutional. Smoothing the input before subsampling
reduces it [@zhang2019making], and an exercise below works it out.

## Padding and Stride

The pooling layers of the frameworks take a window shape, a stride, and a
padding, with the meaning they have for convolutions (that section).
With $p_\textrm{h}$ rows of padding in total and stride $s_\textrm{h}$, a
window of height $k_\textrm{h}$ on an input of height $n_\textrm{h}$ gives an
output of height

$$\left\lfloor\frac{n_\textrm{h}-k_\textrm{h}+p_\textrm{h}}{s_\textrm{h}}\right\rfloor+1,$$

where the fraction counts the steps the window can take and the 1 counts its
starting position. The width follows in the same way. The frameworks expect
four-dimensional input, so
we put the $4\times4$ matrix of the numbers 0 to 15 into a tensor with one
example and one channel.

TensorFlow prefers, and is optimized for, the *channels-last* layout
(batch, height, width, channels).

In [6]:
X = d2l.reshape(d2l.arange(16, dtype=d2l.float32), (1, 4, 4, 1))
X

<tf.Tensor: shape=(1, 4, 4, 1), dtype=float32, numpy=
array([[[[ 0.],
         [ 1.],
         [ 2.],
         [ 3.]],

        [[ 4.],
         [ 5.],
         [ 6.],
         [ 7.]],

        [[ 8.],
         [ 9.],
         [10.],
         [11.]],

        [[12.],
         [13.],
         [14.],
         [15.]]]], dtype=float32)>

Pooling summarizes a region, so the usual stride equals the window shape:
the windows then tile the input without overlapping. A $3\times3$ window on
the $4\times4$ input fits once, and the output is the maximum of the
top-left $3\times3$ block, 10. Pooling has no parameters, so the layer needs
no initialization.

The stride of `tf.keras.layers.MaxPool2D` defaults to the window shape.

In [7]:
pool = tf.keras.layers.MaxPool2D(3)
pool(X)

<tf.Tensor: shape=(1, 1, 1, 1), dtype=float32, numpy=array([[[[10.]]]], dtype=float32)>

Overlapping windows need an explicit stride. With one row and one column of
padding on each side and stride 2, the output is
$\lfloor(4-3+2)/2\rfloor+1 = 2$ entries high and wide.

Keras pooling layers accept only `padding='valid'` (no padding) and
`padding='same'`, which puts any odd row or column of padding at the bottom
and right. For one row and one column on every side we use the function
`tf.nn.max_pool2d`, which takes the padding of each axis explicitly.

In [8]:
pool = partial(tf.nn.max_pool2d, ksize=3, strides=2,
               padding=[[0, 0], [1, 1], [1, 1], [0, 0]])
pool(X)

<tf.Tensor: shape=(1, 2, 2, 1), dtype=float32, numpy=
array([[[[ 5.],
         [ 7.]],

        [[13.],
         [15.]]]], dtype=float32)>

Which value fills the padding? A convolution pads with zeros, which add
nothing to a weighted sum. For a maximum, the neutral value is $-\infty$:
padded positions then never win, and each output is the maximum of the real
entries under its window. Here the windows cover the real entries in rows
and columns $\{0, 1\}$ and $\{1, 2, 3\}$, so the outputs are 5, 7, 13, and
15. Padding with $-\infty$ also keeps a property that every maximum has:
adding a constant to every input entry adds the same constant to every
output. We check it on `X - 16`, whose entries all lie below zero.

In [9]:
pool(X - 16)

<tf.Tensor: shape=(1, 2, 2, 1), dtype=float32, numpy=
array([[[[-11.],
         [ -9.]],

        [[ -3.],
         [ -1.]]]], dtype=float32)>

Each output is 16 below the previous one. Zero padding would have produced
0 in the three windows that touch the padding, since 0 exceeds every real
entry there.

Average pooling has no neutral padding value. If the padded positions count
as zeros in the mean, they pull the outputs at the border toward 0; the
alternative is to average only the real entries under each window.

`tf.keras.layers.AvgPool2D` with `padding='same'` averages only the real
entries under each window.

Pooling windows can also have different heights and widths, here a
$2\times3$ window with stride $(2, 3)$ and one column of padding on the
left and right.

In [10]:
tf.nn.max_pool2d(X, ksize=(2, 3), strides=(2, 3),
                 padding=[[0, 0], [0, 0], [1, 1], [0, 0]])

<tf.Tensor: shape=(1, 2, 2, 1), dtype=float32, numpy=
array([[[[ 5.],
         [ 7.]],

        [[13.],
         [15.]]]], dtype=float32)>

## Multiple Channels

A convolutional layer sums over its input channels: every output channel
reads all of them. A pooling layer instead pools each input channel
separately, so its output has as many channels as its input. To check this,
we concatenate `X` and `X + 1` along the channel axis to obtain an input
with two channels.

In [11]:
X = d2l.concat([X, X + 1], 3)  # the channel axis is last
X

<tf.Tensor: shape=(1, 4, 4, 2), dtype=float32, numpy=
array([[[[ 0.,  1.],
         [ 1.,  2.],
         [ 2.,  3.],
         [ 3.,  4.]],

        [[ 4.,  5.],
         [ 5.,  6.],
         [ 6.,  7.],
         [ 7.,  8.]],

        [[ 8.,  9.],
         [ 9., 10.],
         [10., 11.],
         [11., 12.]],

        [[12., 13.],
         [13., 14.],
         [14., 15.],
         [15., 16.]]]], dtype=float32)>

The pooling layer with padding 1 and stride 2 from above returns two
channels: the first holds the outputs computed earlier, 5, 7, 13, and 15,
and the second the same values plus 1.

In [12]:
pool(X)

<tf.Tensor: shape=(1, 2, 2, 2), dtype=float32, numpy=
array([[[[ 5.,  6.],
         [ 7.,  8.]],

        [[13., 14.],
         [15., 16.]]]], dtype=float32)>

The result is in the channels-last layout, so the two channels appear side
by side in the last axis.

## Summary

Pooling replaces each window of its input by its maximum or its mean. It has
no parameters, it pools each channel separately, and its output shape follows
the padding and stride rules of convolutions; max-pooling pads with
$-\infty$, so padding never changes a maximum. A $2\times2$ window with
stride 2 halves the resolution, which lets every later kernel reach twice as
far, and it absorbs a shift of a response that stays inside its window, but
not one that crosses into the next. Max-pooling keeps the strength of the
strongest response; average pooling dilutes it over the window.

Later networks often halve the resolution with a convolution of stride 2
instead, which learns how to combine the entries of a window rather than
fixing the combination to a maximum or a mean. Pooling then remains in a
second role, *global average pooling*, which averages each channel over all
positions of the final feature map [@Lin.Chen.Yan.2013]. The next
section puts convolutions and pooling together into a complete classifier.

## Exercises

1. [code] **Average pooling as convolution.** Express
   $k_\textrm{h} \times k_\textrm{w}$ average pooling with stride
   $(s_\textrm{h}, s_\textrm{w})$ as a convolution. Give the kernel, and
   state how the convolution must be structured across channels so that
   each channel is pooled separately, as pooling layers do. On the input
   of the figure, verify that your convolution reproduces
   `pool2d(X, (2, 2), 'avg')`.

1. **Receptive field and cost of downsampling.** Consider $3\times3$
   convolutions with stride 1 and padding 1, with 64 input and 64 output
   channels, on a $224\times224$ input.
    1. How many such layers does a stack need before the receptive field
       of a unit at its top covers the whole image?
    1. Now insert $2\times2$ max-pooling with stride 2 after every second
       convolutional layer. How many convolutional layers are needed now?
    1. Count the multiply-adds of the two stacks, and the comparisons that
       the pooling layers perform. Which part of the cost does pooling
       remove, and what does it cost itself?

1. **Pooling alternatives.**
    1. Do we need a separate minimum-pooling layer, or can it be expressed
       through an operation already in this section?
    1. *Softmax pooling* with temperature $\tau > 0$ replaces the window
       entries $x_1, \ldots, x_m$ by their softmax-weighted average
       $$\sum_{i=1}^m x_i \frac{\exp(x_i / \tau)}{\sum_{j=1}^m \exp(x_j / \tau)}.$$
       Find its limits as $\tau \to 0$ and as $\tau \to \infty$. Give one
       cost that softmax pooling adds relative to max-pooling.

1. **Aliasing and blur-pool.** Naive stride-2 downsampling keeps
   every second entry of its input. Measure frequency in cycles per input
   sample, so that the one-dimensional signals $(1, 0, 1, 0, 1, 0)$ and
   $(0, 1, 0, 1, 0, 1)$ both have frequency $\tfrac{1}{2}$.
    1. Apply naive stride-2 downsampling to both signals, which differ by
       a shift of one position, and compare the two outputs. Why is this
       behavior called *aliasing*?
    1. Which input frequencies can a stride-2 subsampler represent
       faithfully?
    1. Blur-pool [@zhang2019making] counteracts aliasing by applying
       a small averaging filter before subsampling. Work out what
       blur-pool with a two-tap box filter
       $(\tfrac{1}{2}, \tfrac{1}{2})$ computes on the two signals above.
       Which pooling operation from this section does it coincide with?

1. [code] **Backward pass of max-pooling.** Derive the gradient of
   two-dimensional max-pooling with respect to its input, for a
   $k_\textrm{h} \times k_\textrm{w}$ window and stride equal to the
   window size, and implement it without automatic differentiation.
   Verify against the gradient that automatic differentiation computes
   through the built-in max-pooling operation of this section on a small
   random tensor, to a relative error below $10^{-4}$.

    *Adapted from Stanford CS231n,
    [Assignment 2](https://cs231n.github.io/assignments2024/assignment2/),
    ConvolutionalNetworks, naive max-pooling backward pass.*

1. **Max-pooling via ReLU.** ●
    1. Prove that max-pooling cannot be implemented by a convolution
       alone.
    1. Express $\max(a, b)$ using only affine operations and
       $\textrm{ReLU}(x) = \max(0, x)$.
    1. Use this to implement max-pooling as a stack of convolutions and
       ReLU layers.
    1. Count the channels per layer and the ReLU layers that your
       construction uses for a $2 \times 2$ window and for a
       $3 \times 3$ window. How does the number of ReLU layers grow with
       the number of entries in the window?

[Discussions](https://d2l.discourse.group/t/274)